In [1]:
import os
import json

# 1. Define the exact directory soccerdata looks for
config_dir = r"C:\Users\John Goh\soccerdata\config"
os.makedirs(config_dir, exist_ok=True)

# 2. Define the custom league mapping
league_mapping = {
  "NED-Eredivisie": {
    "FBref": "Eredivisie"
  },
  "POR-Primeira Liga": {
    "FBref": "Primeira Liga"
  },
  "ENG-Championship": {
    "FBref": "Championship"
  }
}

# 3. Write the JSON file directly to the system
file_path = os.path.join(config_dir, "league_dict.json")
with open(file_path, "w") as f:
    json.dump(league_mapping, f, indent=4)

print(f"Successfully created league config at: {file_path}")

Successfully created league config at: C:\Users\John Goh\soccerdata\config\league_dict.json


In [3]:
import soccerdata as sd
import pandas as pd
from sqlalchemy import create_engine

# 1. Database Connection String
engine = create_engine("mysql+pymysql://root:@localhost:3306/moneyballton")

# 2. Initialize the Scraper
fbref = sd.FBref(
    leagues=["Big 5 European Leagues Combined", "NED-Eredivisie", "POR-Primeira Liga", "ENG-Championship"],
    seasons=[2019, 2020, 2021, 2022, 2023, 2024]
)

# 3. Extract Tables (Standard only)
print("Pulling standard stats...")
df_raw = fbref.read_player_season_stats(stat_type="standard")

# 4. Transform: Clean Multi-Index
# Flatten the multi-index so SQL can read the columns
df_raw = df_raw.reset_index()

# 5. Load directly into MySQL
print("Loading into MySQL database...")
df_raw.to_sql(
    name='fbref_raw_2018_2024', 
    con=engine, 
    if_exists='replace', 
    index=False
)

print("ELT Pipeline Complete!")

[09/06/26 23:03:07] INFO     Saving cached data to C:\Users\John Goh\soccerdata\data\FBref           _common.py:250

[09/06/26 23:03:18] WARNING  C:\Users\John Goh\OneDrive - Singapore Management                      ]8;id=10144512;file://C:\Python312\Lib\warnings.py\warnings.py]8;;\:]8;id=10144513;file://C:\Python312\Lib\warnings.py#112\112]8;;\
                             University\Desktop\Desktop Cleanup\Y4S1 Mods\proj                                     
                             moneyballton\moneyballton\venv\Lib\site-packages\soccerdata\_common.py                
                             :144: UserWarning: Season id "2021" is ambiguous: interpreting as                     
                             "20-21"                                                                               
                               warnings.warn(msg, stacklevel=1)                                                    
                                                                                                                   

Pulling standard stats...
Loading into MySQL database...
ELT Pipeline Complete!
